# ЛР 5, вариант 2: сортировка подсчётом на GPU. Замеры для отчёта

Ноутбук делает всё для раздела «Результаты» отчёта:
1. записывает в Colab программы: сдаваемую `lab5.cu`, профилирующую `lab5_prof.cu` (замеры по фазам через `cudaEvent`), CPU-версию `lab5_cpu.cpp` (`std::sort` и сортировка подсчётом) и генератор данных `gen_data.py`;
2. проверяет корректность всех трёх программ по `numpy.sort` на разных видах данных;
3. строит таблицу «CPU и GPU» для разных n (до 135 млн);
4. показывает время по фазам (копирование, гистограмма, сканирование, запись результата);
5. сравнивает виды данных (случайные, все одинаковые, перекошенные и др.);
6. исследует конфигурации запуска ядер;
7. рисует график и запускает профайлер `nsys` (замена `nvprof`).

**Как запускать:** *Среда выполнения → Сменить тип среды выполнения → T4 GPU*, затем *Среда выполнения → Выполнить все*.
Размеры и число повторов задаются в ячейке «Параметры». Самые большие тесты (100–135 млн) занимают до 540 МБ на диск каждый и несколько минут; их можно убрать из `SIZES`.

In [ ]:
%%shell
nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
nvcc --version | tail -2
lscpu | grep -E "Model name|^CPU\(s\)"
free -g | head -2
df -h /content | tail -1

## 1. Файлы программ

In [ ]:
%%writefile lab5.cu
// ЛР №5, вариант 2. Сортировка подсчётом для int из диапазона [0, 2^24 - 1] (CUDA).
// Гистограмма на атомарных операциях + сканирование Блеллока любого размера
// с рекурсией и бесконфликтным использованием разделяемой памяти.
// Сборка: nvcc -Werror cross-execution-space-call -lm -o lab5 lab5.cu
#include <stdio.h>
#include <stdlib.h>
#include <limits.h>

#define CSC(call)                                                          \
do {                                                                       \
    cudaError_t res = (call);                                              \
    if (res != cudaSuccess) {                                              \
        fprintf(stderr, "ERROR: CUDA failure in %s:%d. Message: %s\n",     \
                __FILE__, __LINE__, cudaGetErrorString(res));              \
        exit(0);                                                           \
    }                                                                      \
} while (0)

#ifndef RANGE_BITS
#define RANGE_BITS 24
#endif
#define RANGE (1 << RANGE_BITS)  // значения лежат в [0, RANGE - 1], RANGE = 2^24

// ----- параметры запуска (фиксированы, не зависят от размера данных) --------
#ifndef SCAN_BLOCK
#define SCAN_BLOCK 512           // потоков в блоке сканирования
#endif
#ifndef SCAN_GRID
#define SCAN_GRID 256            // блоков в сетке сканирования
#endif
#ifndef STREAM_BLOCK
#define STREAM_BLOCK 256
#endif
#ifndef STREAM_GRID
#define STREAM_GRID 1024
#endif
#ifndef CHUNK
#define CHUNK 64                 // длина участка выхода, обрабатываемого одним потоком
#endif

#define ELEMS (2 * SCAN_BLOCK)   // элементов, сканируемых одним блоком
#define LOG_NUM_BANKS 5          // 32 банка разделяемой памяти
// Смещение для устранения конфликтов банков: после каждых 32 элементов
// вставляется один пропуск, обращения вида 2^k * (2t+1) - 1 расходятся по банкам.
#define CFO(x) ((x) >> LOG_NUM_BANKS)

// ----------------------------------------------------------------------------
// Гистограмма: hist[a[i]]++ на атомарных операциях.
// ----------------------------------------------------------------------------
__global__ void histogram(const int *a, int *hist, int n) {
    long long idx = (long long)blockIdx.x * blockDim.x + threadIdx.x;
    long long offset = (long long)blockDim.x * gridDim.x;
    for (long long i = idx; i < n; i += offset)
        atomicAdd(&hist[a[i]], 1);
}

// ----------------------------------------------------------------------------
// Исключающее сканирование Блеллока внутри блока из ELEMS элементов.
// Суммы блоков записываются в sums[]. Блоки обходятся циклом с шагом gridDim.x,
// поэтому число блоков в сетке фиксировано.
// ----------------------------------------------------------------------------
__global__ void scan_blocks(int *data, int *sums, int len, int nblocks) {
    __shared__ int temp[ELEMS + (ELEMS >> LOG_NUM_BANKS)];
    int tid = threadIdx.x;

    for (int b = blockIdx.x; b < nblocks; b += gridDim.x) {
        long long base = (long long)b * ELEMS;
        int ai = tid, bi = tid + SCAN_BLOCK;
        int offA = CFO(ai), offB = CFO(bi);

        temp[ai + offA] = (base + ai < len) ? data[base + ai] : 0;
        temp[bi + offB] = (base + bi < len) ? data[base + bi] : 0;

        // Восходящий проход (свёртка).
        int offset = 1;
        for (int d = ELEMS >> 1; d > 0; d >>= 1) {
            __syncthreads();
            if (tid < d) {
                int i = offset * (2 * tid + 1) - 1;
                int j = offset * (2 * tid + 2) - 1;
                i += CFO(i);
                j += CFO(j);
                temp[j] += temp[i];
            }
            offset <<= 1;
        }

        // Сумма блока -> sums[b], последний элемент обнуляется.
        if (tid == 0) {
            int last = (ELEMS - 1) + CFO(ELEMS - 1);
            sums[b] = temp[last];
            temp[last] = 0;
        }

        // Нисходящий проход.
        for (int d = 1; d < ELEMS; d <<= 1) {
            offset >>= 1;
            __syncthreads();
            if (tid < d) {
                int i = offset * (2 * tid + 1) - 1;
                int j = offset * (2 * tid + 2) - 1;
                i += CFO(i);
                j += CFO(j);
                int t = temp[i];
                temp[i] = temp[j];
                temp[j] += t;
            }
        }
        __syncthreads();

        if (base + ai < len) data[base + ai] = temp[ai + offA];
        if (base + bi < len) data[base + bi] = temp[bi + offB];
        __syncthreads();   // temp будет использован следующим блоком
    }
}

// Добавление просканированных сумм блоков ко всем элементам своих блоков.
__global__ void add_offsets(int *data, const int *sums, int len) {
    long long idx = (long long)blockIdx.x * blockDim.x + threadIdx.x;
    long long offset = (long long)blockDim.x * gridDim.x;
    for (long long i = idx; i < len; i += offset)
        data[i] += sums[i / ELEMS];
}

// Сканирование массива любого размера, рекурсивно:
// 1) сканируем блоки, 2) рекурсивно сканируем их суммы, 3) добавляем смещения.
static void scan(int *dev, int len) {
    int nblocks = (len + ELEMS - 1) / ELEMS;
    int *sums = NULL;
    CSC(cudaMalloc(&sums, sizeof(int) * nblocks));
    scan_blocks<<<SCAN_GRID, SCAN_BLOCK>>>(dev, sums, len, nblocks);
    CSC(cudaGetLastError());
    if (nblocks > 1) {
        scan(sums, nblocks);
        add_offsets<<<STREAM_GRID, STREAM_BLOCK>>>(dev, sums, len);
        CSC(cudaGetLastError());
    }
    CSC(cudaFree(sums));
}

// ----------------------------------------------------------------------------
// Запись отсортированного массива по просканированной гистограмме.
// ex[v] -- число элементов, меньших v (исключающее сканирование гистограммы),
// значит элемент выхода с номером pos равен наибольшему v, для которого ex[v] <= pos.
// Выход разбит на участки по CHUNK элементов: поток бинарным поиском находит
// значение для начала участка и дальше идёт по значениям вперёд. Работа делится
// поровну по выходным позициям, поэтому перекоса нагрузки нет даже если
// почти все числа одинаковы.
// ----------------------------------------------------------------------------
__global__ void write_sorted(int *out, const int *ex, int n) {
    long long idx = (long long)blockIdx.x * blockDim.x + threadIdx.x;
    long long offset = (long long)blockDim.x * gridDim.x;
    long long nchunks = ((long long)n + CHUNK - 1) / CHUNK;

    for (long long c = idx; c < nchunks; c += offset) {
        long long s = c * CHUNK;
        long long e = s + CHUNK;
        if (e > n) e = n;

        int lo = 0, hi = RANGE - 1;                 // наибольшее v: ex[v] <= s
        while (lo < hi) {
            int mid = (lo + hi + 1) >> 1;
            if (ex[mid] <= s) lo = mid; else hi = mid - 1;
        }
        int v = lo;
        int next = (v + 1 < RANGE) ? ex[v + 1] : INT_MAX;
        for (long long pos = s; pos < e; pos++) {
            while (next <= pos) {
                v++;
                next = (v + 1 < RANGE) ? ex[v + 1] : INT_MAX;
            }
            out[pos] = v;
        }
    }
}

int main() {
    int n;
    if (fread(&n, sizeof(int), 1, stdin) != 1) {
        fprintf(stderr, "ERROR: cannot read array length\n");
        return 0;
    }
    if (n < 0) {
        fprintf(stderr, "ERROR: negative array length %d\n", n);
        return 0;
    }
    if (n == 0) return 0;

    size_t bytes = sizeof(int) * (size_t)n;
    int *h = (int *)malloc(bytes);
    if (h == NULL) {
        fprintf(stderr, "ERROR: not enough host memory\n");
        return 0;
    }
    if (fread(h, sizeof(int), (size_t)n, stdin) != (size_t)n) {
        fprintf(stderr, "ERROR: input is shorter than the declared length\n");
        free(h);
        return 0;
    }
    for (int i = 0; i < n; i++) {
        if (h[i] < 0 || h[i] >= RANGE) {
            fprintf(stderr, "ERROR: element %d = %d is outside [0, %d]\n", i, h[i], RANGE - 1);
            free(h);
            return 0;
        }
    }

    int *dev_data = NULL, *dev_hist = NULL;
    CSC(cudaMalloc(&dev_data, bytes));
    CSC(cudaMalloc(&dev_hist, sizeof(int) * (size_t)RANGE));
    CSC(cudaMemcpy(dev_data, h, bytes, cudaMemcpyHostToDevice));
    CSC(cudaMemset(dev_hist, 0, sizeof(int) * (size_t)RANGE));

    histogram<<<STREAM_GRID, STREAM_BLOCK>>>(dev_data, dev_hist, n);
    CSC(cudaGetLastError());

    scan(dev_hist, RANGE);

    // Исходные данные больше не нужны: результат пишется в тот же буфер.
    write_sorted<<<STREAM_GRID, STREAM_BLOCK>>>(dev_data, dev_hist, n);
    CSC(cudaGetLastError());
    CSC(cudaDeviceSynchronize());

    CSC(cudaMemcpy(h, dev_data, bytes, cudaMemcpyDeviceToHost));
    CSC(cudaFree(dev_data));
    CSC(cudaFree(dev_hist));

    fwrite(h, sizeof(int), (size_t)n, stdout);
    fflush(stdout);

    free(h);
    return 0;
}


In [ ]:
%%writefile lab5_prof.cu
// ЛР №5, вариант 2. ПРОФИЛИРУЮЩАЯ версия (для отчёта, не для сдачи).
// Алгоритм тот же, что в lab5.cu, плюс замеры по фазам через cudaEvent и
// настраиваемые конфигурации запуска.
//
// Сборка:  nvcc -arch=sm_75 -O2 -o lab5_prof lab5_prof.cu
// Запуск:  ./lab5_prof [--noout] [--stream-grid G] [--stream-block B] [--scan-grid G] < data.bin
//   --noout          не выводить отсортированный массив
//   --stream-grid/-block  конфигурация ядер histogram, add_offsets, write_sorted (1024 и 256)
//   --scan-grid      число блоков ядра scan_blocks (256); потоков в блоке 512 (константа)
// Результат замеров -- одна строка TIMING=... в stderr.
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <limits.h>
#include <chrono>

#define CSC(call)                                                          \
do {                                                                       \
    cudaError_t res = (call);                                              \
    if (res != cudaSuccess) {                                              \
        fprintf(stderr, "ERROR: CUDA failure in %s:%d. Message: %s\n",     \
                __FILE__, __LINE__, cudaGetErrorString(res));              \
        exit(0);                                                           \
    }                                                                      \
} while (0)

#ifndef RANGE_BITS
#define RANGE_BITS 24
#endif
#define RANGE (1 << RANGE_BITS)

#ifndef SCAN_BLOCK
#define SCAN_BLOCK 512
#endif
#ifndef CHUNK
#define CHUNK 64
#endif
#define ELEMS (2 * SCAN_BLOCK)
#define LOG_NUM_BANKS 5
#define CFO(x) ((x) >> LOG_NUM_BANKS)

// Конфигурации запуска (меняются из командной строки).
static int g_stream_grid = 1024, g_stream_block = 256, g_scan_grid = 256;

__global__ void histogram(const int *a, int *hist, int n) {
    long long idx = (long long)blockIdx.x * blockDim.x + threadIdx.x;
    long long offset = (long long)blockDim.x * gridDim.x;
    for (long long i = idx; i < n; i += offset)
        atomicAdd(&hist[a[i]], 1);
}

__global__ void scan_blocks(int *data, int *sums, int len, int nblocks) {
    __shared__ int temp[ELEMS + (ELEMS >> LOG_NUM_BANKS)];
    int tid = threadIdx.x;
    for (int b = blockIdx.x; b < nblocks; b += gridDim.x) {
        long long base = (long long)b * ELEMS;
        int ai = tid, bi = tid + SCAN_BLOCK;
        int offA = CFO(ai), offB = CFO(bi);
        temp[ai + offA] = (base + ai < len) ? data[base + ai] : 0;
        temp[bi + offB] = (base + bi < len) ? data[base + bi] : 0;
        int offset = 1;
        for (int d = ELEMS >> 1; d > 0; d >>= 1) {
            __syncthreads();
            if (tid < d) {
                int i = offset * (2 * tid + 1) - 1;
                int j = offset * (2 * tid + 2) - 1;
                i += CFO(i);
                j += CFO(j);
                temp[j] += temp[i];
            }
            offset <<= 1;
        }
        if (tid == 0) {
            int last = (ELEMS - 1) + CFO(ELEMS - 1);
            sums[b] = temp[last];
            temp[last] = 0;
        }
        for (int d = 1; d < ELEMS; d <<= 1) {
            offset >>= 1;
            __syncthreads();
            if (tid < d) {
                int i = offset * (2 * tid + 1) - 1;
                int j = offset * (2 * tid + 2) - 1;
                i += CFO(i);
                j += CFO(j);
                int t = temp[i];
                temp[i] = temp[j];
                temp[j] += t;
            }
        }
        __syncthreads();
        if (base + ai < len) data[base + ai] = temp[ai + offA];
        if (base + bi < len) data[base + bi] = temp[bi + offB];
        __syncthreads();
    }
}

__global__ void add_offsets(int *data, const int *sums, int len) {
    long long idx = (long long)blockIdx.x * blockDim.x + threadIdx.x;
    long long offset = (long long)blockDim.x * gridDim.x;
    for (long long i = idx; i < len; i += offset)
        data[i] += sums[i / ELEMS];
}

static void scan(int *dev, int len) {
    int nblocks = (len + ELEMS - 1) / ELEMS;
    int *sums = NULL;
    CSC(cudaMalloc(&sums, sizeof(int) * nblocks));
    scan_blocks<<<g_scan_grid, SCAN_BLOCK>>>(dev, sums, len, nblocks);
    CSC(cudaGetLastError());
    if (nblocks > 1) {
        scan(sums, nblocks);
        add_offsets<<<g_stream_grid, g_stream_block>>>(dev, sums, len);
        CSC(cudaGetLastError());
    }
    CSC(cudaFree(sums));
}

__global__ void write_sorted(int *out, const int *ex, int n) {
    long long idx = (long long)blockIdx.x * blockDim.x + threadIdx.x;
    long long offset = (long long)blockDim.x * gridDim.x;
    long long nchunks = ((long long)n + CHUNK - 1) / CHUNK;
    for (long long c = idx; c < nchunks; c += offset) {
        long long s = c * CHUNK;
        long long e = s + CHUNK;
        if (e > n) e = n;
        int lo = 0, hi = RANGE - 1;
        while (lo < hi) {
            int mid = (lo + hi + 1) >> 1;
            if (ex[mid] <= s) lo = mid; else hi = mid - 1;
        }
        int v = lo;
        int next = (v + 1 < RANGE) ? ex[v + 1] : INT_MAX;
        for (long long pos = s; pos < e; pos++) {
            while (next <= pos) {
                v++;
                next = (v + 1 < RANGE) ? ex[v + 1] : INT_MAX;
            }
            out[pos] = v;
        }
    }
}

// ---- секундомер на cudaEvent -----------------------------------------------
struct Stopwatch { cudaEvent_t a, b; };
static void sw_init(Stopwatch *s) { CSC(cudaEventCreate(&s->a)); CSC(cudaEventCreate(&s->b)); }
static void sw_start(Stopwatch *s) { CSC(cudaEventRecord(s->a, 0)); }
static double sw_stop(Stopwatch *s) {
    CSC(cudaEventRecord(s->b, 0));
    CSC(cudaEventSynchronize(s->b));
    float ms = 0.0f;
    CSC(cudaEventElapsedTime(&ms, s->a, s->b));
    return (double)ms;
}
static double now_ms() {
    using namespace std::chrono;
    return duration<double, std::milli>(steady_clock::now().time_since_epoch()).count();
}

int main(int argc, char **argv) {
    bool noout = false;
    for (int i = 1; i < argc; i++) {
        if (!strcmp(argv[i], "--noout")) noout = true;
        else if (!strcmp(argv[i], "--stream-grid") && i + 1 < argc) g_stream_grid = atoi(argv[++i]);
        else if (!strcmp(argv[i], "--stream-block") && i + 1 < argc) g_stream_block = atoi(argv[++i]);
        else if (!strcmp(argv[i], "--scan-grid") && i + 1 < argc) g_scan_grid = atoi(argv[++i]);
        else { fprintf(stderr, "ERROR: unknown or incomplete argument %s\n", argv[i]); return 0; }
    }
    if (g_stream_grid < 1 || g_scan_grid < 1 || g_stream_block < 1 || g_stream_block > 1024) {
        fprintf(stderr, "ERROR: invalid launch configuration\n");
        return 0;
    }

    CSC(cudaFree(0));   // инициализация контекста вне замеров

    double t0 = now_ms();
    int n;
    if (fread(&n, sizeof(int), 1, stdin) != 1 || n < 0) {
        fprintf(stderr, "ERROR: invalid array length\n");
        return 0;
    }
    if (n == 0) return 0;
    size_t bytes = sizeof(int) * (size_t)n;
    int *h = (int *)malloc(bytes);
    if (h == NULL) { fprintf(stderr, "ERROR: not enough host memory\n"); return 0; }
    if (fread(h, sizeof(int), (size_t)n, stdin) != (size_t)n) {
        fprintf(stderr, "ERROR: input is shorter than the declared length\n");
        return 0;
    }
    for (int i = 0; i < n; i++)
        if (h[i] < 0 || h[i] >= RANGE) {
            fprintf(stderr, "ERROR: element %d = %d is outside [0, %d]\n", i, h[i], RANGE - 1);
            return 0;
        }
    double read_ms = now_ms() - t0;

    int *dev_data = NULL, *dev_hist = NULL;
    CSC(cudaMalloc(&dev_data, bytes));
    CSC(cudaMalloc(&dev_hist, sizeof(int) * (size_t)RANGE));

    Stopwatch sw_total, sw_h2d, sw_hist, sw_scan, sw_write, sw_d2h;
    sw_init(&sw_total); sw_init(&sw_h2d); sw_init(&sw_hist);
    sw_init(&sw_scan); sw_init(&sw_write); sw_init(&sw_d2h);

    sw_start(&sw_total);

    sw_start(&sw_h2d);
    CSC(cudaMemcpy(dev_data, h, bytes, cudaMemcpyHostToDevice));
    CSC(cudaMemset(dev_hist, 0, sizeof(int) * (size_t)RANGE));
    double h2d_ms = sw_stop(&sw_h2d);

    sw_start(&sw_hist);
    histogram<<<g_stream_grid, g_stream_block>>>(dev_data, dev_hist, n);
    CSC(cudaGetLastError());
    double hist_ms = sw_stop(&sw_hist);

    sw_start(&sw_scan);
    scan(dev_hist, RANGE);
    double scan_ms = sw_stop(&sw_scan);

    sw_start(&sw_write);
    write_sorted<<<g_stream_grid, g_stream_block>>>(dev_data, dev_hist, n);
    CSC(cudaGetLastError());
    double write_ms = sw_stop(&sw_write);

    sw_start(&sw_d2h);
    CSC(cudaMemcpy(h, dev_data, bytes, cudaMemcpyDeviceToHost));
    double d2h_ms = sw_stop(&sw_d2h);

    double gpu_total_ms = sw_stop(&sw_total);

    CSC(cudaFree(dev_data));
    CSC(cudaFree(dev_hist));

    double out_ms = 0.0;
    if (!noout) {
        double t1 = now_ms();
        fwrite(h, sizeof(int), (size_t)n, stdout);
        fflush(stdout);
        out_ms = now_ms() - t1;
    }

    fprintf(stderr,
            "TIMING= n=%d stream_grid=%d stream_block=%d scan_grid=%d h2d_ms=%.4f hist_ms=%.4f "
            "scan_ms=%.4f write_ms=%.4f d2h_ms=%.4f gpu_total_ms=%.4f read_ms=%.4f out_ms=%.4f\n",
            n, g_stream_grid, g_stream_block, g_scan_grid, h2d_ms, hist_ms, scan_ms, write_ms,
            d2h_ms, gpu_total_ms, read_ms, out_ms);
    free(h);
    return 0;
}


In [ ]:
%%writefile lab5_cpu.cpp
// ЛР №5, вариант 2. CPU-версия для сравнения с GPU (для отчёта, не для сдачи).
// Замеряет два алгоритма на одном ядре процессора:
//   std::sort            -- обычная сортировка из стандартной библиотеки;
//   сортировка подсчётом -- тот же алгоритм, что на GPU (гистограмма, префиксные суммы, запись).
// Результаты обоих сравниваются между собой.
//
// Сборка:  g++ -O2 -o lab5_cpu lab5_cpu.cpp
// Запуск:  ./lab5_cpu [--noout] < data.bin
// Результат замера -- одна строка CPUTIME=... в stderr.
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <algorithm>
#include <vector>
#include <chrono>

#ifndef RANGE_BITS
#define RANGE_BITS 24
#endif
#define RANGE (1 << RANGE_BITS)

static double now_ms() {
    using namespace std::chrono;
    return duration<double, std::milli>(steady_clock::now().time_since_epoch()).count();
}

int main(int argc, char **argv) {
    bool noout = false;
    for (int i = 1; i < argc; i++) {
        if (!strcmp(argv[i], "--noout")) noout = true;
        else { fprintf(stderr, "ERROR: unknown argument %s\n", argv[i]); return 0; }
    }

    double t0 = now_ms();
    int n;
    if (fread(&n, sizeof(int), 1, stdin) != 1 || n < 0) {
        fprintf(stderr, "ERROR: invalid array length\n");
        return 0;
    }
    if (n == 0) return 0;
    std::vector<int> a(n);
    if (fread(a.data(), sizeof(int), (size_t)n, stdin) != (size_t)n) {
        fprintf(stderr, "ERROR: input is shorter than the declared length\n");
        return 0;
    }
    for (int i = 0; i < n; i++)
        if (a[i] < 0 || a[i] >= RANGE) {
            fprintf(stderr, "ERROR: element %d = %d is outside [0, %d]\n", i, a[i], RANGE - 1);
            return 0;
        }
    double read_ms = now_ms() - t0;

    // 1. std::sort
    std::vector<int> s(a);
    double t1 = now_ms();
    std::sort(s.begin(), s.end());
    double std_sort_ms = now_ms() - t1;

    // 2. Сортировка подсчётом
    std::vector<int> c(a.size());
    double t2 = now_ms();
    std::vector<int> cnt(RANGE, 0);
    for (int i = 0; i < n; i++) cnt[a[i]]++;
    size_t pos = 0;
    for (int v = 0; v < RANGE; v++)
        for (int k = cnt[v]; k > 0; k--) c[pos++] = v;
    double count_sort_ms = now_ms() - t2;

    int same = (s == c) ? 1 : 0;

    double out_ms = 0.0;
    if (!noout) {
        double t3 = now_ms();
        fwrite(s.data(), sizeof(int), (size_t)n, stdout);
        fflush(stdout);
        out_ms = now_ms() - t3;
    }
    fprintf(stderr, "CPUTIME= n=%d std_sort_ms=%.4f count_sort_ms=%.4f same=%d read_ms=%.4f out_ms=%.4f\n",
            n, std_sort_ms, count_sort_ms, same, read_ms, out_ms);
    return 0;
}


In [ ]:
%%writefile gen_data.py
#!/usr/bin/env python3
"""Генератор бинарных тестов для ЛР 5 (формат: int32 n, затем n чисел int32).

Использование:
    python3 gen_data.py N out.bin [--kind KIND] [--seed S] [--bits B]

Виды данных (--kind), значения лежат в [0, 2^B - 1], по умолчанию B = 24:
    random    равномерно случайные
    equal     все числа одинаковы (худший случай для атомарных операций гистограммы)
    skewed    90% одинаковых, остальные случайные
    sorted    уже отсортированы (неубывающие)
    reversed  отсортированы в обратном порядке
    few       всего 16 различных значений
    zeros     все нули

Из Python:
    from gen_data import gen_data
    gen_data('in.bin', 1_000_000, 'random', seed=1)
"""
import argparse
import numpy as np

KINDS = ['random', 'equal', 'skewed', 'sorted', 'reversed', 'few', 'zeros']


def _chunk(kind, start, size, n, R, rng):
    if kind == 'random':
        return rng.integers(0, R, size, dtype=np.int64)
    if kind == 'equal':
        return np.full(size, R // 2, dtype=np.int64)
    if kind == 'skewed':
        a = rng.integers(0, R, size, dtype=np.int64)
        a[rng.random(size) < 0.9] = R // 3
        return a
    if kind in ('sorted', 'reversed'):
        idx = np.arange(start, start + size, dtype=np.int64)
        if kind == 'reversed':
            idx = n - 1 - idx
        return idx * R // n
    if kind == 'few':
        return rng.integers(0, 16, size, dtype=np.int64) * (R // 16)
    if kind == 'zeros':
        return np.zeros(size, dtype=np.int64)
    raise ValueError(f'unknown kind: {kind}')


def gen_data(path, n, kind='random', seed=0, bits=24, block=8_000_000):
    """Записывает файл блоками по `block` чисел (мало памяти даже для n = 135 млн)."""
    R = 1 << bits
    rng = np.random.default_rng(seed)
    with open(path, 'wb') as f:
        f.write(np.array([n], dtype='<i4').tobytes())
        for start in range(0, n, block):
            size = min(block, n - start)
            f.write(_chunk(kind, start, size, n, R, rng).astype('<i4').tobytes())


if __name__ == '__main__':
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument('n', type=int)
    ap.add_argument('out')
    ap.add_argument('--kind', default='random', choices=KINDS)
    ap.add_argument('--seed', type=int, default=0)
    ap.add_argument('--bits', type=int, default=24)
    a = ap.parse_args()
    gen_data(a.out, a.n, a.kind, a.seed, a.bits)
    print(f'{a.kind}: {a.n} int32 values written to {a.out}')


## 2. Компиляция

In [ ]:
%%shell
set -e
nvcc -arch=sm_75 -Werror cross-execution-space-call -lm -o lab5 lab5.cu
nvcc -arch=sm_75 -O2 -o lab5_prof lab5_prof.cu
g++ -O2 -o lab5_cpu lab5_cpu.cpp
ls -l lab5 lab5_prof lab5_cpu

## 3. Параметры и вспомогательные функции

Виды данных (`gen_data.py`): `random`, `equal` (все числа одинаковы), `skewed` (90% одинаковых), `sorted`, `reversed`, `few` (16 значений), `zeros`.
Файлы создаются один раз и повторно не генерируются.

In [ ]:
import subprocess, re, os, time, statistics
import numpy as np
import pandas as pd
from gen_data import gen_data, KINDS

# ---------------- Параметры ----------------
SIZES     = [1_000_000, 5_000_000, 10_000_000, 50_000_000, 100_000_000, 135_000_000]
CPU_MAX_N = 135_000_000   # CPU-версию запускаем до этого n
REPS      = 3             # повторов для GPU, берётся медиана
N_STUDY   = 50_000_000    # размер данных для сравнения видов данных и конфигураций
BITS      = 24            # значения из [0, 2^BITS - 1]
# --------------------------------------------

def data_path(kind, n):
    return f'data_{kind}_{n}_b{BITS}.bin'

def ensure(kind, n):
    p = data_path(kind, n)
    if not os.path.exists(p):
        t = time.time()
        gen_data(p, n, kind, seed=n % 1000, bits=BITS)
        print(f'{p}: создан за {time.time() - t:.1f} с, {os.path.getsize(p) / 1e6:.0f} МБ')
    return p

KV = re.compile(r'(\w+)=(-?\d+(?:\.\d+)?(?:e[+-]?\d+)?)\b')

def run(cmd, path):
    with open(path, 'rb') as f:
        r = subprocess.run(cmd, stdin=f, capture_output=True)
    err = r.stderr.decode(errors='replace')
    if 'ERROR' in err:
        raise RuntimeError(f"{' '.join(cmd)}: {err.strip()}")
    return r.stdout, err

def metrics(err, tag):
    line = [l for l in err.splitlines() if l.startswith(tag)][-1]
    return {k: float(v) for k, v in KV.findall(line)}

def median(v):
    return statistics.median(v)

def md_table(df):
    cols = list(df.columns)
    out = ['| ' + ' | '.join(cols) + ' |', '|' + '|'.join(['---'] * len(cols)) + '|']
    for row in df.itertuples(index=False):
        out.append('| ' + ' | '.join('' if pd.isna(v) else (f'{v:.2f}' if isinstance(v, (float, np.floating)) else str(v)) for v in row) + ' |')
    return '\n'.join(out)

for n in SIZES:
    ensure('random', n)

## 4. Проверка корректности

Все три программы (`lab5`, `lab5_prof`, `lab5_cpu`) сравниваются с `numpy.sort` побайтово на всех видах данных и на длинах вокруг границ участков (63, 64, 65 элементов).

In [ ]:
CHECK_SIZES = [1, 63, 64, 65, 100_000, 1_000_000]
rows = []
all_ok = True
for kind in KINDS:
    row = {'вид данных': kind}
    for n in CHECK_SIZES:
        p = ensure(kind, n)
        a = np.fromfile(p, dtype='<i4')[1:]
        exp = np.sort(a).astype('<i4').tobytes()
        ok = True
        for cmd in (['./lab5'], ['./lab5_prof'], ['./lab5_cpu']):
            out, _ = run(cmd, p)
            ok &= (out == exp)
        row[f'n={n}'] = 'OK' if ok else 'FAIL'
        all_ok &= ok
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))
print('\nВсе проверки пройдены:', all_ok)

## 5. Таблица «CPU и GPU»

* **GPU ядра**: гистограмма + сканирование + запись результата (события `cudaEvent`, без копирования).
* **GPU total**: то же вместе с копированием массива на видеокарту и обратно.
* **CPU std::sort** и **CPU подсчётом**: сортировка на одном ядре процессора (`g++ -O2`), сортировка подсчётом реализована тем же алгоритмом, что на GPU.
* Чтение и запись файлов в замеры не входят.

In [ ]:
rows = []
for n in SIZES:
    p = ensure('random', n)
    run(['./lab5_prof', '--noout'], p)   # прогрев
    g = [metrics(run(['./lab5_prof', '--noout'], p)[1], 'TIMING=') for _ in range(REPS)]
    row = {'n': n,
           'GPU ядра, мс':  median([x['hist_ms'] + x['scan_ms'] + x['write_ms'] for x in g]),
           'GPU total, мс': median([x['gpu_total_ms'] for x in g])}
    if n <= CPU_MAX_N:
        reps_cpu = 3 if n <= 10_000_000 else 1
        c = [metrics(run(['./lab5_cpu', '--noout'], p)[1], 'CPUTIME=') for _ in range(reps_cpu)]
        row['CPU std::sort, мс'] = median([x['std_sort_ms'] for x in c])
        row['CPU подсчётом, мс'] = median([x['count_sort_ms'] for x in c])
        row['Ускорение к std::sort (ядра)'] = row['CPU std::sort, мс'] / row['GPU ядра, мс']
        row['Ускорение к std::sort (total)'] = row['CPU std::sort, мс'] / row['GPU total, мс']
        row['Ускорение к подсчётом (total)'] = row['CPU подсчётом, мс'] / row['GPU total, мс']
    rows.append(row)
    print(f'n={n:,} готово')
table = pd.DataFrame(rows)
table.to_csv('table_cpu_gpu.csv', index=False)
print(table.round(2).to_string(index=False))
print('\n--- Markdown для отчёта ---')
print(md_table(table))

## 6. На что тратится время (по фазам)

Доли от полного времени работы на GPU (`gpu_total`). Гистограмма и запись результата зависят от n, а сканирование всегда обрабатывает 2^24 элемента, поэтому его время почти постоянно.

In [ ]:
rows = []
for n in SIZES:
    d = metrics(run(['./lab5_prof', '--noout'], ensure('random', n))[1], 'TIMING=')
    parts = {'копирование на GPU': d['h2d_ms'], 'гистограмма': d['hist_ms'], 'сканирование': d['scan_ms'],
             'запись результата': d['write_ms'], 'копирование с GPU': d['d2h_ms']}
    row = {'n': n, 'всего, мс': d['gpu_total_ms']}
    row.update({k + ', мс': v for k, v in parts.items()})
    row.update({k + ', %': 100.0 * v / d['gpu_total_ms'] for k, v in parts.items()})
    rows.append(row)
phases = pd.DataFrame(rows)
phases.to_csv('table_phases.csv', index=False)
ms_cols = ['n', 'всего, мс'] + [c for c in phases.columns if c.endswith(', мс') and c != 'всего, мс']
pc_cols = ['n'] + [c for c in phases.columns if c.endswith(', %')]
print(phases[ms_cols].round(2).to_string(index=False))
print()
print(phases[pc_cols].round(1).to_string(index=False))
print('\n--- Markdown для отчёта ---')
print(md_table(phases[ms_cols])); print(); print(md_table(phases[pc_cols]))

## 7. Влияние вида данных

Сравнение при одном и том же n (`N_STUDY`). Ожидается, что на `equal` и `skewed` гистограмма медленнее: много потоков одновременно делают `atomicAdd` в один и тот же счётчик.
Запись результата по участкам одинаково быстро работает на любых данных, потому что работа делится по позициям выхода, а не по значениям.

In [ ]:
rows = []
for kind in KINDS:
    p = ensure(kind, N_STUDY)
    run(['./lab5_prof', '--noout'], p)
    g = [metrics(run(['./lab5_prof', '--noout'], p)[1], 'TIMING=') for _ in range(REPS)]
    rows.append({'вид данных': kind,
                 'гистограмма, мс': median([x['hist_ms'] for x in g]),
                 'сканирование, мс': median([x['scan_ms'] for x in g]),
                 'запись, мс': median([x['write_ms'] for x in g]),
                 'GPU total, мс': median([x['gpu_total_ms'] for x in g])})
kinds_df = pd.DataFrame(rows)
kinds_df.to_csv('table_kinds.csv', index=False)
print(f'n = {N_STUDY:,}')
print(kinds_df.round(2).to_string(index=False))
print('\n--- Markdown для отчёта ---')
print(md_table(kinds_df))

## 8. Конфигурации запуска

Ядра `histogram`, `add_offsets` и `write_sorted` запускаются с `<<<grid, block>>>`. Число потоков фиксировано внутри запуска, а цикл с шагом покрывает любой размер данных.
Для ядра сканирования перебирается число блоков (потоков в блоке 512, оно задаёт размер разделяемой памяти).

In [ ]:
p = ensure('random', N_STUDY)
STREAM = [(16, 256), (64, 256), (256, 256), (1024, 256), (4096, 256),
          (1024, 64), (1024, 128), (1024, 512), (1024, 1024)]
rows = []
for gr, bl in STREAM:
    d = metrics(run(['./lab5_prof', '--noout', '--stream-grid', str(gr), '--stream-block', str(bl)], p)[1], 'TIMING=')
    rows.append({'grid': gr, 'block': bl, 'всего потоков': gr * bl,
                 'гистограмма, мс': d['hist_ms'], 'запись, мс': d['write_ms']})
stream_df = pd.DataFrame(rows)
stream_df.to_csv('table_stream_cfg.csv', index=False)
print(f'n = {N_STUDY:,}')
print(stream_df.round(2).to_string(index=False))
print('\n--- Markdown для отчёта ---'); print(md_table(stream_df))

SCAN_GRIDS = [1, 4, 16, 64, 256, 1024]
rows = []
for sg in SCAN_GRIDS:
    d = metrics(run(['./lab5_prof', '--noout', '--scan-grid', str(sg)], p)[1], 'TIMING=')
    rows.append({'блоков в сетке': sg, 'потоков в блоке': 512, 'сканирование, мс': d['scan_ms']})
scan_df = pd.DataFrame(rows)
scan_df.to_csv('table_scan_cfg.csv', index=False)
print()
print(scan_df.round(3).to_string(index=False))
print('\n--- Markdown для отчёта ---'); print(md_table(scan_df))

## 9. График

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
t = table
if 'CPU std::sort, мс' in t:
    ax[0].plot(t['n'], t['CPU std::sort, мс'], 'o-', label='CPU std::sort')
    ax[0].plot(t['n'], t['CPU подсчётом, мс'], 'd-', label='CPU подсчётом')
ax[0].plot(t['n'], t['GPU ядра, мс'], 's-', label='GPU (ядра)')
ax[0].plot(t['n'], t['GPU total, мс'], '^-', label='GPU (с копированием)')
ax[0].set_xscale('log'); ax[0].set_yscale('log'); ax[0].set_xlabel('n'); ax[0].set_ylabel('время, мс')
ax[0].set_title('Сортировка n чисел'); ax[0].grid(True, which='both', alpha=0.3); ax[0].legend()
if 'Ускорение к std::sort (total)' in t:
    s = t.dropna(subset=['Ускорение к std::sort (total)'])
    ax[1].plot(s['n'], s['Ускорение к std::sort (ядра)'], 's-', label='к std::sort (ядра)')
    ax[1].plot(s['n'], s['Ускорение к std::sort (total)'], '^-', label='к std::sort (с копированием)')
    ax[1].plot(s['n'], s['Ускорение к подсчётом (total)'], 'd-', label='к CPU подсчётом (с копированием)')
    ax[1].axhline(1.0, color='gray', ls='--')
    ax[1].set_xscale('log'); ax[1].set_xlabel('n'); ax[1].set_ylabel('ускорение, раз')
    ax[1].set_title('Ускорение GPU относительно CPU'); ax[1].grid(True, which='both', alpha=0.3); ax[1].legend()
plt.tight_layout(); plt.savefig('time_vs_n.png', dpi=150); plt.show()

## 10. Профайлер (замена `nvprof`)

`nvprof` в современных CUDA не поддерживается, вместо него используются Nsight Systems (`nsys`) и Nsight Compute (`ncu`).
Если ячейка пишет, что `nsys` не найден, в этой среде он не установлен: в отчёте можно сослаться на замеры через `cudaEvent` из разделов 5–8.

In [ ]:
!which nsys ncu nvprof || echo "nsys / ncu / nvprof не найдены в PATH"
!nsys profile --stats=true --force-overwrite true -o prof_report ./lab5_prof --noout < {data_path('random', N_STUDY)} 2>&1 | head -70

## 11. Что вставить в отчёт

* Таблица из раздела 5 (CPU и GPU, ускорение) и график `time_vs_n.png`.
* Таблицы из раздела 6 (фазы) и раздела 7 (виды данных); объяснение, почему растёт время гистограммы на одинаковых данных (конфликты атомарных операций на одном адресе).
* Таблицы из раздела 8 (конфигурации запуска).
* Вывод `nsys` из раздела 10 (если доступен).
* Описание оборудования из первой ячейки.

Файлы `table_*.csv` и `time_vs_n.png` можно скачать из панели слева.